# 5. SupportSense — Business Impact & Rollout Recommendation
## Business question: Under what conditions would a support assistant be worth piloting?

This notebook translates operational assumptions into scenarios. Because the repository does not contain real ticket volumes, agent wages, handling times, adoption rates, or measured AI effects, every business input below is an **editable assumption**, not a finding. The goal is to identify the variables that matter and the evidence needed before making a go/no-go decision.

## 1. Define transparent planning assumptions
Edit these values when reliable company data becomes available. `eligible_tickets_per_month` is the monthly volume that could plausibly use the assistant, not total support demand.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT=next((p for p in [Path.cwd(),Path.cwd().parent] if (p/"data").exists()),None)
if ROOT is None: raise FileNotFoundError("Run from project root or notebooks/ folder.")

assumptions = {
    "eligible_tickets_per_month": 5000,
    "loaded_agent_cost_per_hour": 28.0,
    "baseline_handling_minutes": 12.0,
    "minutes_saved_per_assisted_ticket": 2.0,
    "adoption_rate": 0.35,
    "resolution_quality_pass_rate": 0.90,
    "monthly_platform_cost": 1200.0,
    "monthly_implementation_cost_amortized": 500.0,
}
pd.DataFrame({"assumption":assumptions.keys(),"value":assumptions.values()})

## 2. Calculate the base-case capacity and cost scenario
Capacity released is not automatically cash savings. It is potential agent time freed for other work, conditional on adoption and on the assumed time saving being real.

In [ ]:
a=assumptions
assisted_tickets=a["eligible_tickets_per_month"]*a["adoption_rate"]
quality_adjusted_tickets=assisted_tickets*a["resolution_quality_pass_rate"]
hours_released=quality_adjusted_tickets*a["minutes_saved_per_assisted_ticket"]/60
capacity_value=hours_released*a["loaded_agent_cost_per_hour"]
monthly_cost=a["monthly_platform_cost"]+a["monthly_implementation_cost_amortized"]
net_capacity_value=capacity_value-monthly_cost
base_case=pd.DataFrame([
    ("Eligible tickets / month",a["eligible_tickets_per_month"]),
    ("Assisted tickets / month",assisted_tickets),
    ("Quality-adjusted assisted tickets",quality_adjusted_tickets),
    ("Agent hours potentially released",hours_released),
    ("Illustrative capacity value ($)",capacity_value),
    ("Estimated monthly cost ($)",monthly_cost),
    ("Illustrative net capacity value ($)",net_capacity_value),
],columns=["measure","value"])
display(base_case.round(2))

## 3. Scenario analysis
The low/base/high cases are planning scenarios, not confidence bounds. Their inputs are deliberately visible so stakeholders can challenge them.

In [ ]:
scenarios = pd.DataFrame([
    {"scenario":"Low","adoption":0.15,"minutes_saved":0.5,"quality_pass":0.80,"monthly_cost":1800},
    {"scenario":"Base","adoption":0.35,"minutes_saved":2.0,"quality_pass":0.90,"monthly_cost":1700},
    {"scenario":"High","adoption":0.60,"minutes_saved":3.0,"quality_pass":0.95,"monthly_cost":2200},
])
scenarios["assisted_tickets"]=a["eligible_tickets_per_month"]*scenarios["adoption"]
scenarios["hours_released"]=scenarios["assisted_tickets"]*scenarios["quality_pass"]*scenarios["minutes_saved"]/60
scenarios["capacity_value"]=scenarios["hours_released"]*a["loaded_agent_cost_per_hour"]
scenarios["net_capacity_value"]=scenarios["capacity_value"]-scenarios["monthly_cost"]
scenarios["break_even_minutes_saved"]=scenarios["monthly_cost"]*60/(
    scenarios["assisted_tickets"]*scenarios["quality_pass"]*a["loaded_agent_cost_per_hour"]
)
display(scenarios.round(2))

In [ ]:
fig,ax=plt.subplots(figsize=(8,4))
ax.bar(scenarios["scenario"],scenarios["net_capacity_value"])
ax.axhline(0,linewidth=1)
ax.set_title("Illustrative monthly net capacity value by scenario")
ax.set_ylabel("Value less assumed monthly cost ($)")
ax.grid(axis="y",alpha=.25)
plt.tight_layout()
plt.show()

## 4. Sensitivity: which assumptions matter most?
This grid shows how adoption and time saved combine. It holds quality pass rate and costs constant at base-case assumptions.

In [ ]:
adoption_grid=np.array([0.10,0.20,0.35,0.50,0.65,0.80])
minutes_grid=np.array([0.5,1.0,1.5,2.0,2.5,3.0,4.0])
sens=[]
for adoption in adoption_grid:
    for minutes in minutes_grid:
        hours=a["eligible_tickets_per_month"]*adoption*a["resolution_quality_pass_rate"]*minutes/60
        value=hours*a["loaded_agent_cost_per_hour"]-monthly_cost
        sens.append({"adoption_rate":adoption,"minutes_saved":minutes,"net_capacity_value":value})
sensitivity=pd.DataFrame(sens)
pivot=sensitivity.pivot(index="minutes_saved",columns="adoption_rate",values="net_capacity_value")
display(pivot.round(0))
fig,ax=plt.subplots(figsize=(9,5))
im=ax.imshow(pivot.values,aspect="auto",origin="lower")
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels([f"{x:.0%}" for x in pivot.columns])
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
ax.set_xlabel("Adoption rate"); ax.set_ylabel("Minutes saved per assisted ticket")
ax.set_title("Sensitivity of illustrative net capacity value ($)")
fig.colorbar(im,ax=ax,label="Net capacity value ($)")
plt.tight_layout()
plt.show()

## 5. Operational and safety guardrails
Business value should only be considered alongside service quality and risk. These are proposed pilot guardrails; set final thresholds with support and security stakeholders.

In [ ]:
guardrails=pd.DataFrame([
    ("Answer correctness","Human-reviewed rubric","No material regression vs baseline"),
    ("Groundedness / citations","Reviewer verifies source supports claim","No unsupported policy claims"),
    ("Unsafe disclosure","Security review of sensitive-data cases","Zero tolerance for credential disclosure"),
    ("Unnecessary abstention","Share of answerable questions escalated","Monitor by topic; investigate increases"),
    ("Escalation quality","Correct routing and useful handoff context","No increase in unresolved escalations"),
    ("Latency","p50 and p95 end-to-end latency","Set threshold based on workflow needs"),
    ("Agent adoption","Usage among eligible cases","Measure separately from effectiveness"),
],columns=["guardrail","measurement","proposed decision rule"])
display(guardrails)

## 6. Pilot roadmap and decision gates

In [ ]:
roadmap=pd.DataFrame([
    (1,"Data readiness","Collect representative, de-identified support cases and approved knowledge sources","Data owner + Support Ops"),
    (2,"Offline validation","Expand benchmark; label correct answers, citations, abstention, and safety","Support SMEs + ML"),
    (3,"Shadow mode","Generate suggestions without showing them to customers; review failure modes","Support Ops + Security"),
    (4,"Controlled pilot","Randomized rollout with pre-registered primary metric and guardrails","Product + Analytics"),
    (5,"Scale decision","Compare outcomes, cost, latency, risk, and subgroup performance","Business owner"),
],columns=["phase","gate","exit criterion","accountable group"])
display(roadmap)

## 7. Recommendation

**Recommendation: proceed to a measured validation pilot only after expanding and human-reviewing the evaluation set. Do not claim ROI or deploy broadly on the current synthetic benchmark alone.**

A sensible decision sequence is:
1. **Now:** use the project as a reproducible prototype and portfolio demonstration.
2. **Before shadow mode:** add representative support examples, document versions, source permissions, and a reviewer rubric.
3. **Before live pilot:** agree on primary quality metric, safety guardrails, randomization unit, sample-size calculation, and monitoring owner.
4. **Before scaling:** replace every planning assumption with observed ticket volume, adoption, handling-time difference, quality pass rate, and fully loaded operating cost.

### Limitations
- All business scenario inputs are editable assumptions.
- Capacity value is not guaranteed cash savings; realizing it depends on staffing and workflow.
- The offline benchmark is synthetic and too small to establish customer outcomes.
- The A/B notebook compares policies offline and is not a live randomized experiment.

This closes the analysis sequence: EDA → segmentation → retrieval modeling → offline experiment → business recommendation.